<a href="https://colab.research.google.com/github/deburky/language-models/blob/main/gpt-oss-claude-code/colab-finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune gpt-oss-20b on Tool-Use Examples

Trains a LoRA adapter on your custom `data/train.jsonl` (pre-rendered gpt-oss harmony format).  
Requires **A100 40GB** minimum, **H100 80GB** recommended.

Tested with Colab Pro on `NVIDIA RTX PRO 6000 Blackwell Server Edition 102.0 GB`.

**Before running:** upload `data/train.jsonl` and `data/valid.jsonl` to this Colab session  
(drag & drop into the file browser, or mount Google Drive).

Inspired by: [Fine-tuning with gpt-oss and Hugging Face Transformers](https://developers.openai.com/cookbook/articles/gpt-oss/fine-tune-transfomers)

Authored by: [Denis Burakov](https://www.github.com/deburky)

## 1. Install dependencies

In [1]:
%pip install torch --index-url https://download.pytorch.org/whl/cu128 -q
%pip install 'trl>=0.20.0' 'peft>=0.17.0' 'transformers>=4.55.0' datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.5 MB/s eta 0:00:00


In [2]:
import torch
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

NVIDIA RTX PRO 6000 Blackwell Server Edition
102.0 GB


## 2. Hugging Face login (needed to download gpt-oss-20b and push adapter)

In [3]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 3. Load dataset

Option A — files uploaded directly to Colab (default).  
Option B — mount Google Drive and point `DATA_DIR` to your drive path.

In [5]:
# Option B: uncomment to mount Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/gpt-oss-data'

DATA_DIR = '/content'  # default: files uploaded to session root

from datasets import load_dataset

train_dataset = load_dataset('json', data_files=f'{DATA_DIR}/train.jsonl', split='train')
eval_dataset  = load_dataset('json', data_files=f'{DATA_DIR}/valid.jsonl', split='train')

print(f'Train: {len(train_dataset)} examples')
print(f'Valid: {len(eval_dataset)} examples')
print('\nSample (last 300 chars):')
print(train_dataset[0]['text'][-300:])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 284 examples
Valid: 71 examples

Sample (last 300 chars):
onald Fisher in the 1920s. It is consistent (converges to the true value with enough data) and asymptotically efficient (achieves the Cramér-Rao lower bound). Its weakness is that it can overfit with small samples and it ignores prior knowledge, which is where Bayesian estimation picks up.<|return|>


## 4. Load model

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Mxfp4Config

MODEL_ID = 'openai/gpt-oss-20b'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

quantization_config = Mxfp4Config(dequantize=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    attn_implementation='eager',
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map='auto',
)
print('Model loaded.')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Model loaded.


## 5. Configure LoRA

In [7]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules='all-linear',
    target_parameters=[
        '7.mlp.experts.gate_up_proj',
        '7.mlp.experts.down_proj',
        '15.mlp.experts.gate_up_proj',
        '15.mlp.experts.down_proj',
        '23.mlp.experts.gate_up_proj',
        '23.mlp.experts.down_proj',
    ],
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

trainable params: 15,040,512 || all params: 20,929,797,696 || trainable%: 0.0719


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:212: UserWarning: Unsupported layer type '<class 'transformers.models.gpt_oss.modeling_gpt_oss.GptOssExperts'>' encountered, proceed at your own risk.
  warnings.warn(f"Unsupported layer type '{type(module)}' encountered, proceed at your own risk.", UserWarning)


## 6. Train

In [10]:
from huggingface_hub import login
import getpass

token = getpass.getpass("HF Write Token: ")
login(token=token)

HF Write Token: ··········


In [11]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = 'gpt-oss-claude-code'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    # Data — already rendered, skip chat template
    dataset_text_field='text',
    max_length=2048,
    # Optimizer
    learning_rate=1e-4,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    # Stability
    gradient_checkpointing=True,
    bf16=True,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    # Logging / saving
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=25,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id='gpt-oss-claude-code',
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998}.


Step,Training Loss,Validation Loss
25,1.408188,1.154293
50,0.527239,0.521854
75,0.413348,0.484062
100,0.455352,0.478477


TrainOutput(global_step=108, training_loss=0.9632699633086169, metrics={'train_runtime': 285.41, 'train_samples_per_second': 2.985, 'train_steps_per_second': 0.378, 'total_flos': 3.310947250997069e+16, 'train_loss': 0.9632699633086169})

In [24]:
import re
import json
import glob as globlib
import pathlib

TOOL_SYSTEM = """You are a coding assistant with access to the file system.

Available tools:
- read(path, offset, limit): Read a file
- glob(pat): Find files by pattern
- grep(pat, path): Search for text in files

To use a tool, format it EXACTLY like this:
<tool_call>{"tool": "name", "args": {"key": "value"}}</tool_call>"""


def run_tool(name, args):
    if name == 'glob':
        pat = args.get('pat', '*')
        files = globlib.glob(pat, recursive=True)
        return '\n'.join(sorted(files)) or 'none'
    if name == 'read':
        p = pathlib.Path(args['path'])
        lines = p.read_text().splitlines()
        offset = int(args.get('offset', 0))
        limit = int(args.get('limit', 20))
        return ''.join(f"{offset+i+1:4}| {l}\n" for i, l in enumerate(lines[offset:offset+limit]))
    if name == 'grep':
        pattern = re.compile(args.get('pat', ''))
        p = pathlib.Path(args.get('path', '.'))
        hits = []
        for fp in ([p] if p.is_file() else p.rglob('*')):
            if fp.is_file():
                hits.extend(
                    f'{fp}:{i}:{line.rstrip()}'
                    for i, line in enumerate(
                        fp.read_text(errors='replace').splitlines(), 1
                    )
                    if pattern.search(line)
                )
        return '\n'.join(hits[:40]) or 'none'
    return f'error: unknown tool {name}'


def parse_native_calls(text):
    calls = []
    # Native tag format: <glob>{...}</glob>, <read>{...}</read>, <grep>{...}</grep>
    for tag in ['glob', 'read', 'grep']:
        for m in re.finditer(rf'<{tag}>(.*?)</{tag}>', text, re.DOTALL):
            try:
                calls.append({'name': tag, 'args': json.loads(m.group(1))})
            except:
                pass
    # tool_call format: <tool_call>{"tool": "name", "args": {...}}</tool_call>
    for m in re.finditer(r'<tool_call>(.*?)</tool_call>', text, re.DOTALL):
        try:
            d = json.loads(m.group(1))
            if d.get('tool') in ('glob', 'read', 'grep'):
                calls.append({'name': d['tool'], 'args': d.get('args', {})})
        except Exception:
            pass
    return calls


def parse_raw_to_message(raw):
    """Convert raw gpt-oss output into a proper assistant message dict."""
    thinking, content = '', ''
    if '<|channel|>analysis<|message|>' in raw:
        part = raw.split('<|channel|>analysis<|message|>')[1]
        thinking = part.split('<|end|>')[0].strip()
    if '<|channel|>final<|message|>' in raw:
        part = raw.split('<|channel|>final<|message|>')[1]
        content = re.split(r'<\|end\|>|<\|return\|>', part)[0].strip()
    else:
        content = re.sub(r'<\|[^>]+\|>', '', raw).strip()
    msg = {'role': 'assistant', 'content': content}
    if thinking:
        msg['thinking'] = thinking
    return msg


def chat(user_msg, max_steps=5):
    messages = [
        {'role': 'system', 'content': TOOL_SYSTEM},
        {'role': 'user', 'content': user_msg},
    ]
    for step in range(max_steps):
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True,
            return_tensors='pt', return_dict=True,
        ).to(peft_model.device)
        with torch.no_grad():
            output_ids = peft_model.generate(
                **inputs, max_new_tokens=256, temperature=0.3, do_sample=True
            )
        raw = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=False)
        print(f'[step {step+1}] RAW:', raw[:200])

        calls = parse_native_calls(raw)
        if not calls:
            if '<|channel|>final<|message|>' in raw:
                raw = raw.split('<|channel|>final<|message|>')[-1]
            print('FINAL:', re.sub(r'<\|[^>]+\|>', '', raw).strip())
            break

        for call in calls:
            result = run_tool(call['name'], call['args'])
            print(f'  → {call["name"]}({call["args"]})\n  {result[:200]}')
            messages.append(parse_raw_to_message(raw))
            messages.append({'role': 'user', 'content': f'Tool {call["name"]} returned:\n{result}'})


# Test
chat('Read the first 20 lines of /etc/hostname.')

[step 1] RAW: <|channel|>analysis<|message|>Read file.<|end|><|start|>assistant<|channel|>final<|message|><tool_call>{"tool": "read", "args": {"path": "/etc/hostname", "offset": 0, "limit": 20}}</tool_call>

<|end|
  → read({'path': '/etc/hostname', 'offset': 0, 'limit': 20})
     1| 1a8f6eecadb6

[step 2] RAW: <|channel|>final<|message|>The hostname is 1a8f6eecadb6. It looks like a random string, possibly from a Docker container or a VM. If you need a more descriptive hostname, consider setting it in /etc/h
FINAL: The hostname is 1a8f6eecadb6. It looks like a random string, possibly from a Docker container or a VM. If you need a more descriptive hostname, consider setting it in /etc/hostname and /etc/hosts.


## 7. Save adapter and push to Hub

In [25]:
trainer.save_model(OUTPUT_DIR)
trainer.push_to_hub()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...de-code/training_args.bin: 100%|##########| 5.65kB / 5.65kB            

  ...laude-code/tokenizer.json: 100%|##########| 27.9MB / 27.9MB            

  ...adapter_model.safetensors: 100%|##########| 60.2MB / 60.2MB            

...

  ...de-code/training_args.bin: 100%|##########| 5.65kB / 5.65kB            

  ...laude-code/tokenizer.json: 100%|##########| 27.9MB / 27.9MB            

  ...adapter_model.safetensors: 100%|##########| 60.2MB / 60.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/deburky/gpt-oss-claude-code/commit/971afd7cc602ee7e44d6e226a7cbbda8b6752973', commit_message='End of training', commit_description='', oid='971afd7cc602ee7e44d6e226a7cbbda8b6752973', pr_url=None, repo_url=RepoUrl('https://huggingface.co/deburky/gpt-oss-claude-code', endpoint='https://huggingface.co', repo_type='model', repo_id='deburky/gpt-oss-claude-code'), pr_revision=None, pr_num=None)

## 8. Inference (merge adapter + base model)

> Restart kernel before running this section to free GPU memory.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained('openai/gpt-oss-20b')

base_model = AutoModelForCausalLM.from_pretrained(
    'openai/gpt-oss-20b',
    attn_implementation='eager',
    torch_dtype=torch.bfloat16,
    use_cache=True,
    device_map='auto',
)

model = PeftModel.from_pretrained(base_model, 'deburky/gpt-oss-claude-code')
model = model.merge_and_unload()
print('Model ready.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
MXFP4 quantization requires Triton and kernels installed: CUDA requires Triton >= 3.4.0, XPU requires Triton >= 3.5.0, we will default to dequantizing the model to bf16


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:212: UserWarning: Unsupported layer type '<class 'transformers.models.gpt_oss.modeling_gpt_oss.GptOssExperts'>' encountered, proceed at your own risk.
  warnings.warn(f"Unsupported layer type '{type(module)}' encountered, proceed at your own risk.", UserWarning)


adapter_model.safetensors:   0%|          | 0.00/60.2M [00:00<?, ?B/s]

Model ready.


In [5]:
import re

TOOL_SYSTEM = """You are a coding assistant with access to the file system.

Available tools:
- read(path, offset, limit): Read a file
- glob(pat): Find files by pattern
- grep(pat, path): Search for text in files

To use a tool, format it EXACTLY like this:
<tool_call>{"tool": "name", "args": {"key": "value"}}</tool_call>"""

messages = [
  {'role': 'system', 'content': TOOL_SYSTEM},
  {'role': 'user', 'content': 'List all files in the gpt-oss-claude-code directory.'}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors='pt',
    return_dict=True,
).to(model.device)

output_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.3, do_sample=True)
response = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=False)

if '<|channel|>final<|message|>' in response:
    response = response.split('<|channel|>final<|message|>')[-1]
response = re.sub(r'<\|[^>]+\|>', '', response).strip()
print(response)

<tool_call>{"tool": "glob", "args": {"pat": "gpt-oss-claude-code/**"}}
